# ActiSense – Kaggle training (activities + sports + postures)

Runs the heavy part of ActiSense on Kaggle: pose extraction from the public datasets and training of
1. the **activity model** (LSTM, `har_model.pt`): KTH + own recordings + Penn Action sports/exercises
2. the **posture model** (`pose_model.pt`): yoga + Bharatanatyam postures, plus a `none` class

**Before running:** Settings → Internet **On**, Accelerator **None** (everything runs on CPU; saves GPU quota).
Inputs: `actisense-data` (private), `penn-action-dataset`, `yoga-poses-dataset`, `bharatnatyam-dance-poses`.
Use **Save Version → Save & Run All**; afterwards download the files from the **Output** tab → `outputs/`.

In [ ]:
# 1. Get the code and a Python 3.11 environment with exactly the laptop's MediaPipe version
%cd /kaggle/working
!rm -rf actisense && git clone -q https://github.com/SHARMI-P/actisense.git && git -C actisense log --oneline -1
!pip install -q uv
!uv venv -q -p 3.11 /tmp/py311
!uv pip install -q --python /tmp/py311/bin/python "mediapipe==0.10.14" "numpy<2" opencv-contrib-python scikit-learn matplotlib pandas scipy
!uv pip install -q --python /tmp/py311/bin/python torch --index-url https://download.pytorch.org/whl/cpu || uv pip install -q --python /tmp/py311/bin/python torch
!/tmp/py311/bin/python -c "import sys, mediapipe as mp, torch; print('python', sys.version.split()[0], '| mediapipe', mp.__version__, '| torch', torch.__version__)"


In [ ]:
# 2. Locate the attached datasets
import os
def find(*cands):
    for c in cands:
        if os.path.isdir(c):
            return c
    raise SystemExit('Not found - add the dataset with Add Input: ' + cands[0])
I = '/kaggle/input'
PENN   = find(f'{I}/datasets/kaushalbora18/penn-action-dataset/Penn_Action', f'{I}/penn-action-dataset/Penn_Action')
YOGA   = find(f'{I}/datasets/niharika41298/yoga-poses-dataset/DATASET', f'{I}/yoga-poses-dataset/DATASET')
BHARAT = find(f'{I}/datasets/pranavmanoj/bharatnatyam-dance-poses/new_Dataset', f'{I}/bharatnatyam-dance-poses/new_Dataset')
ACTI   = find(f'{I}/datasets/sharmipandiyan09/actisense-data', f'{I}/actisense-data')
for k, v in dict(PENN=PENN, YOGA=YOGA, BHARAT=BHARAT, ACTI=ACTI).items():
    print(f'{k:7s}', v)
PY = '/tmp/py311/bin/python'
os.environ['GLOG_minloglevel'] = '2'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
TEST = 'person(02|03|05|06|07|08|09|10|22)_|pennTest_'   # KTH official test people + Penn official test videos


In [ ]:
# 3. Penn Action -> 30-frame pose windows (the slow step: roughly 20-60 min on 4 cores)
!{PY} actisense/extract_penn.py --root "{PENN}" --out data_penn --workers 4 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'


In [ ]:
# 4. Posture images -> skeleton features (a few minutes)
!{PY} actisense/extract_images.py --root "{YOGA}/TRAIN" --split train --source yoga --out poses_yoga_train.npz 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'
!{PY} actisense/extract_images.py --root "{YOGA}/TEST" --split test --source yoga --out poses_yoga_test.npz 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'
!{PY} actisense/extract_images.py --root "{BHARAT}" --strip " Augmented" --source bharatanatyam --out poses_bharat.npz 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'


In [ ]:
# 5. Activity model v4: KTH + own recordings + Penn Action (same honest split and v3 settings)
!{PY} actisense/train.py --data "{ACTI}" data_penn --test-pattern "{TEST}" --features xyz --augment --tag v4 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'


In [ ]:
# 6. Posture model: yoga + Bharatanatyam + 'none' (frames taken from the activity windows)
!{PY} actisense/train_pose.py --poses poses_yoga_train.npz poses_yoga_test.npz poses_bharat.npz --negatives "{ACTI}" data_penn --test-pattern "{TEST}" 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'


In [ ]:
# 7. Collect everything to download (Output tab -> outputs/)
import shutil, glob
from IPython.display import Image, display
os.makedirs('outputs', exist_ok=True)
shutil.copy('har_model_v4.pt', 'outputs/har_model.pt')          # the live app loads har_model.pt
for f in ['har_model_v4.pt', 'pose_model.pt', 'results_v4.txt', 'results_pose.txt',
          'confusion_matrix_v4.png', 'confusion_matrix_pose.png']:
    shutil.copy(f, 'outputs/')
shutil.make_archive('outputs/extracted_features', 'zip', '.', 'data_penn')    # reuse later without re-extracting
for f in glob.glob('poses_*.npz'):
    shutil.copy(f, 'outputs/')
print(open('outputs/results_v4.txt').read())
print(open('outputs/results_pose.txt').read())
display(Image('outputs/confusion_matrix_v4.png'))
display(Image('outputs/confusion_matrix_pose.png'))
print(sorted(os.listdir('outputs')))
